# Data Quality Assessment

This notebook conducts a structured data quality assessment of the NovaCred credit application dataset used in automated loan decision-making. The objective is to evaluate whether the dataset provides a reliable and consistent foundation for credit approval decisions, downstream modelling, transparent decision processes, and internal governance and quality control.

This notebook represents the detailed technical analysis for the **data quality pillar** of the governance audit. While the README consolidates overall conclusions across all governance dimensions, this document provides the structured findings, risk assessments, and recommended controls underlying those conclusions.

## Table of Contents

1. [Executive Summary](#1-executive-summary)
2. [Assessment Framework](#2-assessment-framework)
3. [Methodological Principles](#3-methodological-principles)
4. [Dataset Overview](#4-dataset-overview)
5. [Completeness Assessment](#5-completeness-assessment)
   - 5.1 [Global Missingness Overview](#51-global-missingness-overview)
   - 5.2 [Grouped Completeness Analysis](#52-grouped-completeness-analysis)
   - 5.3 [Schema-Level Field Mapping](#53-schema-level-field-mapping)
   - 5.4 [Conditional Completeness of Decision Outcome Fields](#54-conditional-completeness-of-decision-outcome-fields)
   - 5.5 [Spending Behavior Completeness](#55-spending-behavior-completeness)
   - 5.6 [Co-occurrence of Missing Values](#56-co-occurrence-of-missing-values)
   - 5.7 [Completeness Risk Assessment](#57-completeness-risk-assessment)
6. [Consistency Assessment](#6-consistency-assessment)
   - 6.1 [Record Uniqueness](#61-record-uniqueness)
   - 6.2 [Categorical Encoding](#62-categorical-encoding)
   - 6.3 [Date Format Consistency](#63-date-format-consistency)
   - 6.4 [Data Type Consistency](#64-data-type-consistency)
   - 6.5 [Consistency Risk Assessment](#65-consistency-risk-assessment)
7. [Validity Assessment](#7-validity-assessment)
   - 7.1 [Numerical Range Validation](#71-numerical-range-validation)
   - 7.2 [Format Validation](#72-format-validation)
   - 7.3 [Data Type Validation](#73-data-type-validation)
   - 7.4 [Validity Risk Assessment](#74-validity-risk-assessment)
8. [Accuracy and Plausibility Assessment](#8-accuracy-and-plausibility-assessment)
   - 8.1 [Age Plausibility](#81-age-plausibility)
   - 8.2 [Spending vs Income Plausibility](#82-spending-vs-income-plausibility)
   - 8.3 [Decision Logic Plausibility](#83-decision-logic-plausibility)
   - 8.4 [Interest Rate Plausibility](#84-interest-rate-plausibility)
   - 8.5 [Accuracy Risk Assessment](#85-accuracy-risk-assessment)
9. [Consolidated Risk Assessment](#9-consolidated-risk-assessment)
10. [Recommended Controls and Remediation Plan](#10-recommended-controls-and-remediation-plan)
    - 10.1 [Remediation Demonstrations](#101-remediation-demonstrations)
    - 10.2 [Priority Actions](#102-priority-actions)
11. [Export Cleaned Dataset](#11-export-cleaned-dataset)

## 1. Executive Summary

This notebook presents a structured data quality assessment of the NovaCred credit application dataset, evaluating 502 records across 35 fields along four governance dimensions: completeness, consistency, validity, and accuracy. The objective is to determine whether the dataset provides a reliable and auditable foundation for automated loan decision-making.

**Overall dataset risk: Moderate–High**

The assessment identifies 16 distinct data quality issues, of which 6 are classified as high severity. While core credit decision fields are structurally sound - financial variables are nearly complete, conditional decision logic is internally consistent, and most format validations pass — several findings represent material control failures with direct implications for underwriting integrity and regulatory defensibility.

The most critical findings are:

1. **Three loans were approved without documented income.** Income is a core underwriting variable; its absence at the point of decision indicates a control gap in the approval workflow that cannot be remediated retrospectively.
2. **Two application IDs are duplicated**, affecting 4 records. Primary key violations undermine traceability, audit reliability, and the integrity of any downstream joins or aggregations.
3. **Empty strings in identity and demographic fields are not detected as missing** by standard validation checks, causing true incompleteness to be systematically undercounted across `email`, `gender`, `date_of_birth`, and `zip_code`.
4. **87.6% of records lack a processing timestamp**, severely weakening the audit trail for the credit decision lifecycle.

Beyond these high-severity findings, the dataset exhibits moderate issues including an `annual_income` / `annual_salary` schema inconsistency affecting 5 records, inconsistent gender encoding across 111 records (22.1%), mixed data types in `annual_income`, and a single DTI value exceeding the valid domain constraint in an approved application.

**Remediation feasibility is high.** All identified issues are technically addressable through deterministic pipeline controls, as demonstrated in Section 10. Income reconciliation, gender standardisation, deduplication, and decision logic validation can each be implemented without data loss. Priority actions are grouped into four tiers, with P1 items — blocking approvals without income, enforcing primary key constraints, and normalising empty strings — requiring immediate intervention before the dataset is used for model training or regulatory reporting.

Until P1 and P2 controls are implemented, the dataset should be treated as requiring manual review for flagged records and should not be used as a standalone basis for automated credit decisions.

## 2. Assessment Framework

The assessment is structured along four established data quality dimensions:

1. **Completeness** — Whether required attributes are present and populated, particularly those relevant for creditworthiness evaluation and decision integrity.
2. **Consistency** — Internal coherence across fields, including duplicate identifiers, contradictory values, and structurally inconsistent records.
3. **Validity** — Conformity of values to expected formats, ranges, and domain constraints (e.g., financial ratios, identifiers, categorical encodings).
4. **Accuracy** — Whether values appear realistic when assessed in relation to other variables and domain expectations.

For each identified issue, we report:

| Attribute | Description |
|---|---|
| Affected records | Count and percentage of dataset |
| Severity | Low · Moderate · High · Critical |
| Governance impact | Effect on decision integrity, auditability, or model reliability |
| Remediation | Recommended corrective measures |

Issues are classified using the following severity scale:

| Level | Definition |
|---|---|
| **Low** | Minor data hygiene issue with limited practical impact |
| **Moderate** | Relevant issue requiring correction but unlikely to materially affect decisions |
| **High** | Issue likely to affect decision integrity, model reliability, or auditability |
| **Critical** | Strong indication of control failure with material impact on decision processes |

## 3. Methodological Principles

| Principle | Description |
|---|---|
| **Reproducibility** | All calculations use fixed reference points where applicable |
| **Quantification** | Findings are expressed in both absolute counts and percentages |
| **Materiality** | Issues are evaluated not only statistically but in terms of practical impact on decision-making |
| **Structured reporting** | Each assessment section concludes with a standardized risk assessment table summarizing findings and proposed controls |

## 4. Dataset Overview

This section establishes baseline context regarding dataset size, structure, variable types, and the distribution of key decision outcomes. These observations serve as a foundation for the subsequent quality assessments and ensure that later findings are interpreted in the correct structural context.

In [54]:
# importing necessary libraries and modules
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.data_loading import load_raw_data

In [55]:
# loading the raw data
df = load_raw_data("../data/raw/raw_credit_applications.json")
print(f"Dataset shape: {df.shape[0]} records × {df.shape[1]} columns")

Dataset shape: 502 records × 35 columns


In [56]:
# structural overview: columns, data types, and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 35 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            502 non-null    str    
 1   processing_timestamp          62 non-null     str    
 2   loan_purpose                  50 non-null     str    
 3   notes                         2 non-null      str    
 4   full_name                     502 non-null    str    
 5   email                         502 non-null    str    
 6   ssn                           497 non-null    str    
 7   ip_address                    497 non-null    str    
 8   gender                        501 non-null    str    
 9   date_of_birth                 501 non-null    str    
 10  zip_code                      501 non-null    str    
 11  annual_income                 497 non-null    object 
 12  credit_history_months         502 non-null    int64  
 13  debt_to_income  

In [57]:
# summary statistics for numerical fields
df.describe()

,credit_history_months,debt_to_income,savings_balance,spending_shopping,spending_rent,spending_alcohol,spending_dining,spending_healthcare,interest_rate,approved_amount,...,spending_entertainment,spending_insurance,spending_travel,spending_transportation,spending_utilities,spending_groceries,spending_education,spending_adult_entertainment,spending_gambling,annual_salary
count,502.000000,502.000000,502.000000,54.00000,59.000000,11.000000,66.000000,68.000000,292.000000,292.000000,...,72.000000,68.000000,80.000000,61.000000,76.000000,65.000000,64.000000,5.000000,7.000000,5.000000
mean,50.402390,0.246195,29493.503984,456.37037,523.559322,477.090909,472.787879,451.367647,4.564726,47845.890411,...,496.013889,473.176471,493.400000,439.032787,498.605263,487.815385,499.656250,591.200000,457.142857,69200.000000
std,31.234824,0.136296,16775.309756,234.72853,236.866634,221.196046,237.168198,250.460049,1.162866,18103.754530,...,240.582479,230.782442,250.949014,241.350435,246.217036,242.465841,255.760008,200.966913,242.843865,22664.950915
min,-10.000000,0.050000,-5000.000000,73.00000,81.000000,194.000000,75.000000,51.000000,2.500000,15000.000000,...,52.000000,54.000000,54.000000,52.000000,62.000000,112.000000,54.000000,322.000000,125.000000,45000.000000
25%,27.250000,0.150000,17258.250000,269.50000,323.500000,247.000000,266.000000,232.500000,3.500000,34000.000000,...,291.250000,310.750000,255.000000,288.000000,295.750000,254.000000,268.750000,497.000000,277.500000,46000.000000
50%,48.000000,0.230000,27385.500000,443.00000,526.000000,447.000000,462.000000,464.000000,4.550000,48000.000000,...,489.000000,503.000000,532.000000,385.000000,494.500000,503.000000,528.000000,579.000000,444.000000,75000.000000
75%,72.000000,0.350000,38251.500000,648.50000,705.500000,672.000000,650.250000,603.500000,5.600000,62250.000000,...,715.750000,671.500000,703.500000,663.000000,729.750000,691.000000,713.750000,710.000000,662.500000,86000.000000
max,133.000000,1.850000,88078.000000,887.00000,897.000000,757.000000,897.000000,897.000000,6.500000,80000.000000,...,883.000000,890.000000,888.000000,899.000000,881.000000,885.000000,889.000000,848.000000,751.000000,94000.000000


In [58]:
# decision outcome distribution
approved = (df["loan_approved"] == True).sum()
rejected = (df["loan_approved"] == False).sum()
total = len(df)

print(f"Approved: {approved} ({approved/total*100:.1f}%)")
print(f"Rejected: {rejected} ({rejected/total*100:.1f}%)")
print(f"Total:    {total}")

Approved: 292 (58.2%)
Rejected: 210 (41.8%)
Total:    502


In [59]:
# check for undocumented fields not present in the original schema
documented_fields = [
    "id", "full_name", "email", "ssn", "ip_address", "gender", "date_of_birth", "zip_code",
    "annual_income", "credit_history_months", "debt_to_income", "savings_balance",
    "loan_approved", "interest_rate", "approved_amount", "rejection_reason"
]
spending_cols = [c for c in df.columns if c.startswith("spending_")]
undocumented = [c for c in df.columns if c not in documented_fields and c not in spending_cols]

print("Undocumented fields found:")
for col in undocumented:
    non_null = df[col].notna().sum()
    print(f"  {col}: {non_null}/{total} records populated ({non_null/total*100:.1f}%)")

Undocumented fields found:
  processing_timestamp: 62/502 records populated (12.4%)
  loan_purpose: 50/502 records populated (10.0%)
  notes: 2/502 records populated (0.4%)
  annual_salary: 5/502 records populated (1.0%)


**Findings**

The dataset contains 502 credit application records across 35 columns, representing a flattened version of nested JSON records that include applicant information, financial attributes, spending behavior categories, and loan decision outcomes.

The decision outcome is moderately balanced, with 292 approved applications (58.2%) and 210 rejections (41.8%). This distribution provides a stable baseline for downstream bias and fairness analysis.

**Structural observations from the descriptive statistics:**

1. `annual_income` field is stored as `object` type rather than numeric, indicating the presence of string-encoded values that will require closer inspection in the validity assessment
2. `credit_history_months` reports a minimum of −10, which is logically implausible and will be investigated under validity 
3. `savings_balance` includes a minimum of −5,000, which is unusual for a balance field and warrants plausibility review. These issues will be systematically quantified in the relevant assessment sections below

**Undocumented fields:**

Four fields were found in the dataset that do not appear in the documented schema: `processing_timestamp` (62 records), `loan_purpose` (50 records), `notes` (2 records), and `annual_salary` (5 records). The presence of `annual_salary` alongside the documented `annual_income` field is particularly notable and will be investigated in the completeness assessment, as it may indicate a schema inconsistency affecting income data availability.

---

## 5. Completeness Assessment

This section evaluates the presence of required values across all relevant fields. The objective is to identify incomplete records that may compromise decision integrity, model reliability, or audit traceability.

Completeness is assessed with particular focus on:

- Decision-critical financial variables required for creditworthiness evaluation
- Applicant identity and demographic attributes needed for traceability and fairness analysis
- Conditional completeness of decision outcome fields based on approval status
- Structural integrity of spending behavior data after flattening
- Co-occurrence of missing values across multiple fields within individual records

**Important methodological note:** The dataset contains both `NaN` values and empty strings (`""`) in string-type fields. Standard `isna()` checks do not detect empty strings. We therefore normalize empty strings to `NaN` before conducting the completeness analysis to ensure accurate missingness counts.

In [60]:
# audit empty strings vs NaN before normalization
# standard isna() does not detect empty strings — this cell documents their presence
string_fields = ["email", "gender", "date_of_birth", "zip_code", "full_name", "ssn", "ip_address"]

empty_audit = pd.DataFrame({
    "nan_count": df[string_fields].isna().sum(),
    "empty_string_count": df[string_fields].apply(lambda col: (col == "").sum()),
}).assign(
    total_missing=lambda x: x["nan_count"] + x["empty_string_count"],
    total_missing_pct=lambda x: (x["total_missing"] / len(df) * 100).round(2)
)

# show only fields with at least one empty string
affected = empty_audit[empty_audit["empty_string_count"] > 0]

print("Fields containing empty strings (not detected by isna()):")
display(affected)

Fields containing empty strings (not detected by isna()):


,nan_count,empty_string_count,total_missing,total_missing_pct
email,0,7,7,1.39
gender,1,2,3,0.60
date_of_birth,1,4,5,1.00
zip_code,1,1,2,0.40


In [61]:
# normalize empty strings to NaN for accurate completeness measurement
# this affects: email (7), gender (2), date_of_birth (4), zip_code (1)
string_cols = df.select_dtypes(include=["object", "string"]).columns
df[string_cols] = df[string_cols].replace(r"^\s*$", np.nan, regex=True)

### 5.1 Global Missingness Overview

In [62]:
# global missingness overview across all columns
missing_overview = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "non_null_count": df.notna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
}).sort_values("missing_pct", ascending=False)

print("Global missingness overview:")
display(missing_overview)

Global missingness overview:


,missing_count,non_null_count,missing_pct
notes,500,2,99.60
spending_adult_entertainment,497,5,99.00
annual_salary,497,5,99.00
spending_gambling,495,7,98.61
spending_alcohol,491,11,97.81
loan_purpose,452,50,90.04
spending_shopping,448,54,89.24
spending_rent,443,59,88.25
spending_transportation,441,61,87.85
processing_timestamp,440,62,87.65


**Findings**

The global overview reveals that many columns report high missingness rates, with more than half of all columns exceeding 80%. However, this pattern is primarily an artifact of the spending behavior structure: each applicant has between 1 and 4 spending categories populated, meaning that the 15 individual `spending_*` columns are inherently sparse by design rather than due to data collection failures.

Outside the spending variables, `notes` (99.6%), `annual_salary` (99.0%), `loan_purpose` (90.0%), and `processing_timestamp` (87.6%) exhibit high missingness. These are all undocumented fields not present in the original schema, suggesting they were inconsistently collected or added to the pipeline at a later stage.

To assess materiality appropriately, we group variables into governance-relevant categories for focused analysis.

### 5.2 Grouped Completeness Analysis

In [63]:
# define field groups for focused completeness checks
field_groups = {
    "Decision-critical financials": [
        "annual_income", "debt_to_income", "credit_history_months", "savings_balance"
    ],
    "Decision outcome fields": [
        "loan_approved", "interest_rate", "approved_amount", "rejection_reason"
    ],
    "Identity and traceability": [
        "id", "full_name", "ssn", "email", "ip_address"
    ],
    "Demographic attributes": [
        "gender", "date_of_birth", "zip_code"
    ]
}

In [64]:
# missingness summary per field group
for group_name, fields in field_groups.items():
    group_missing = pd.DataFrame({
        "missing_count": df[fields].isna().sum(),
        "non_null_count": df[fields].notna().sum(),
        "missing_pct": (df[fields].isna().mean() * 100).round(2)
    }).sort_values("missing_pct", ascending=False)
    
    print(f"\n{group_name}:")
    display(group_missing)


Decision-critical financials:


,missing_count,non_null_count,missing_pct
annual_income,5,497,1.0
debt_to_income,0,502,0.0
credit_history_months,0,502,0.0
savings_balance,0,502,0.0



Decision outcome fields:


,missing_count,non_null_count,missing_pct
rejection_reason,292,210,58.17
interest_rate,210,292,41.83
approved_amount,210,292,41.83
loan_approved,0,502,0.00



Identity and traceability:


,missing_count,non_null_count,missing_pct
email,7,495,1.39
ip_address,5,497,1.00
ssn,5,497,1.00
id,0,502,0.00
full_name,0,502,0.00



Demographic attributes:


,missing_count,non_null_count,missing_pct
date_of_birth,5,497,1.0
gender,3,499,0.6
zip_code,2,500,0.4


**Findings**

Decision-critical financial fields are nearly complete. `annual_income` is missing for 5 records (1.0%), while `debt_to_income`, `credit_history_months`, and `savings_balance` are fully populated. The 5 missing `annual_income` values correspond exactly to the 5 records where the undocumented `annual_salary` field is populated instead — indicating a schema inconsistency rather than true data loss. The income information exists but was recorded under a different field name.

Identity and traceability fields show targeted gaps: `ssn` and `ip_address` are each missing for 5 records (1.0%), and `email` is effectively missing for 7 records (1.4%) due to empty strings. The `id` and `full_name` fields are fully populated.

Demographic attributes show minor gaps: `gender` is missing for 3 records (0.6%) combining 1 NaN and 2 empty strings, `date_of_birth` for 5 records (1.0%) combining 1 NaN and 4 empty strings, and `zip_code` for 2 records (0.4%).

Decision outcome fields show no unexpected missingness — `interest_rate` and `approved_amount` are missing for exactly the 210 rejected applications, and `rejection_reason` is missing for exactly the 292 approved applications. This alignment is verified in the conditional completeness check below.

### 5.3 Schema-Level Field Mapping

In [65]:
# investigate the relationship between annual_income and annual_salary
has_salary = df["annual_salary"].notna()
has_income = df["annual_income"].notna()

print(f"Records with annual_income only:  {(has_income & ~has_salary).sum()}")
print(f"Records with annual_salary only:  {(~has_income & has_salary).sum()}")
print(f"Records with both:                {(has_income & has_salary).sum()}")
print(f"Records with neither:             {(~has_income & ~has_salary).sum()}")
print()
print("Records using annual_salary:")
display(df.loc[has_salary, ["id", "annual_income", "annual_salary"]])

Records with annual_income only:  497
Records with annual_salary only:  5
Records with both:                0
Records with neither:             0

Records using annual_salary:


,id,annual_income,annual_salary
76,app_436,NaN,45000.0
94,app_421,NaN,46000.0
99,app_479,NaN,94000.0
141,app_463,NaN,86000.0
149,app_449,NaN,75000.0


**Findings**

The 5 records missing `annual_income` are exactly the 5 records where `annual_salary` is populated instead. No record has both fields, and no record is missing income data entirely. This confirms a schema inconsistency: the same financial attribute was recorded under two different field names, likely due to a change in the data collection pipeline or inconsistent field mapping across intake systems.

From a governance perspective, this is a moderate-severity issue. The underlying data is not lost, but any analysis relying solely on `annual_income` would incorrectly treat these 5 applicants as having missing income, potentially leading to exclusion from creditworthiness evaluation or model training.

### 5.4 Conditional Completeness of Decision Outcome Fields

In [66]:
# conditional completeness checks for decision outcome fields
n_total = len(df)
n_approved = int((df["loan_approved"] == True).sum())
n_rejected = int((df["loan_approved"] == False).sum())

checks = [
    ("Approved but approved_amount is missing",
     df.query("loan_approved == True and approved_amount.isna()").shape[0], n_approved),
    ("Approved but interest_rate is missing",
     df.query("loan_approved == True and interest_rate.isna()").shape[0], n_approved),
    ("Approved but rejection_reason is present",
     df.query("loan_approved == True and rejection_reason.notna()").shape[0], n_approved),
    ("Rejected but rejection_reason is missing",
     df.query("loan_approved == False and rejection_reason.isna()").shape[0], n_rejected),
    ("Rejected but interest_rate is present",
     df.query("loan_approved == False and interest_rate.notna()").shape[0], n_rejected),
    ("Rejected but approved_amount is present",
     df.query("loan_approved == False and approved_amount.notna()").shape[0], n_rejected),
]

conditional_df = pd.DataFrame(checks, columns=["Rule", "Affected Records", "Group Size"])
conditional_df["Affected %"] = (conditional_df["Affected Records"] / conditional_df["Group Size"] * 100).round(2)

print(f"Denominators: total={n_total}, approved={n_approved}, rejected={n_rejected}")
display(conditional_df)

Denominators: total=502, approved=292, rejected=210


,Rule,Affected Records,Group Size,Affected %
0,Approved but approved_amount is missing,0,292,0.0
1,Approved but interest_rate is missing,0,292,0.0
2,Approved but rejection_reason is present,0,292,0.0
3,Rejected but rejection_reason is missing,0,210,0.0
4,Rejected but interest_rate is present,0,210,0.0
5,Rejected but approved_amount is present,0,210,0.0


**Findings**

All conditional completeness checks pass with zero violations. Approved records consistently contain `interest_rate` and `approved_amount` without a `rejection_reason`, while rejected records consistently contain a `rejection_reason` without approval-specific fields. This indicates well-structured data capture logic in the core credit decision workflow.

### 5.5 Spending Behavior Completeness

In [67]:
# spending data completeness — record-level coverage
spending_fields = [c for c in df.columns if c.startswith("spending_")]

spending_per_row = df[spending_fields].notna().sum(axis=1)

coverage = pd.DataFrame({
    "Metric": [
        "Records with any spending data",
        "Records with no spending data",
        "Average categories per record",
        "Median categories per record",
        "Maximum categories per record",
    ],
    "Value": [
        int((spending_per_row > 0).sum()),
        int((spending_per_row == 0).sum()),
        round(float(spending_per_row.mean()), 2),
        float(spending_per_row.median()),
        int(spending_per_row.max()),
    ],
    "% of Total": [
        round((spending_per_row > 0).mean() * 100, 2),
        round((spending_per_row == 0).mean() * 100, 2),
        None, None, None
    ]
})

print("Spending data coverage summary:")
display(coverage)

Spending data coverage summary:


,Metric,Value,% of Total
0,Records with any spending data,502.00,100.0
1,Records with no spending data,0.00,0.0
2,Average categories per record,1.65,NaN
3,Median categories per record,1.00,NaN
4,Maximum categories per record,4.00,NaN


In [68]:
# per-category missingness
spending_missing = pd.DataFrame({
    "missing_count": df[spending_fields].isna().sum(),
    "non_null_count": df[spending_fields].notna().sum(),
    "missing_pct": (df[spending_fields].isna().mean() * 100).round(2)
}).sort_values("missing_pct", ascending=False)

print("Per-category missingness:")
display(spending_missing)

Per-category missingness:


,missing_count,non_null_count,missing_pct
spending_adult_entertainment,497,5,99.00
spending_gambling,495,7,98.61
spending_alcohol,491,11,97.81
spending_shopping,448,54,89.24
spending_rent,443,59,88.25
spending_transportation,441,61,87.85
spending_education,438,64,87.25
spending_groceries,437,65,87.05
spending_dining,436,66,86.85
spending_healthcare,434,68,86.45


**Findings**

All 502 records contain at least some spending data, with a median of 2 categories populated per applicant and a maximum of 4. The high per-column missingness (84–99% per category) is therefore a structural characteristic of the data collection process rather than a data quality failure — each applicant reports spending in a small subset of the 15 available categories.

Notably, `spending_gambling` (7 records) and `spending_adult_entertainment` (5 records) have very low population rates. These categories may raise governance concerns regarding the collection of sensitive behavioral data, which will be addressed in the privacy assessment (Notebook 03).

### 5.6 Co-occurrence of Missing Values

In [69]:
# co-occurrence analysis: how many critical fields are missing per record
critical_fields = ["annual_income", "ssn", "ip_address", "email", "gender", "date_of_birth", "zip_code"]

missing_per_record = df[critical_fields].isna().sum(axis=1)

co_occurrence = missing_per_record.value_counts().sort_index().to_frame("record_count")
co_occurrence.index.name = "missing_fields"
co_occurrence["pct"] = (co_occurrence["record_count"] / len(df) * 100).round(2)

print("Distribution of missing critical fields per record:")
display(co_occurrence)

Distribution of missing critical fields per record:


,record_count,pct
missing_fields,,
0,489,97.41
1,7,1.39
2,1,0.20
4,3,0.60
5,1,0.20
6,1,0.20


In [70]:
# inspect records with 3 or more missing critical fields
high_missing = df[missing_per_record >= 3][["id"] + critical_fields]

print(f"Records with 3+ missing critical fields ({len(high_missing)} records):")
display(high_missing)

Records with 3+ missing critical fields (5 records):


,id,annual_income,ssn,ip_address,email,gender,date_of_birth,zip_code
26,app_075,61000,NaN,NaN,NaN,NaN,NaN,NaN
275,app_120,103000,NaN,NaN,NaN,Female,NaN,90240
297,app_268,112000,NaN,NaN,NaN,NaN,1971-05-14,10005
455,app_001,102000,NaN,NaN,stephanie.nguyen47@mail.com,NaN,NaN,NaN
462,app_165,66000,NaN,NaN,NaN,Male,NaN,10077


**Findings**

The majority of records (489 of 502) have no missing critical fields. However, 5 records are missing 3 or more critical fields simultaneously. These clustered gaps suggest systematic data collection failures for specific applicants rather than random missingness. Records with this level of incompleteness may not be suitable for reliable credit evaluation and should be flagged for manual review.

This pattern is important from a governance perspective: isolated missing values can often be managed through imputation or exclusion rules, but records with clustered missingness across identity, demographic, and financial fields indicate a process-level failure in the data intake pipeline.

### 5.7 Completeness Risk Assessment

| Finding | Evidence | Severity | Governance Impact | Recommended Control |
|---------|----------|----------|-------------------|---------------------|
| Missing `annual_income` due to schema inconsistency (`annual_salary`) | 5 records (1.0%) | **Moderate** | Income excluded from evaluation if not reconciled | Reconcile fields; implement schema validation at ingestion |
| Empty strings in identity/demographic fields not detected as missing | 14 affected values across `email`, `gender`, `date_of_birth`, `zip_code` | **High** | Missingness undercounted; downstream analyses operate on incomplete data | Normalize empty strings to null at ingestion; add input validation |
| Missing `processing_timestamp` | 440 records (87.6%) | **High** | Weak audit trail for processing lifecycle | Implement mandatory timestamp logging for all records |
| Missing `loan_purpose` | 452 records (90.0%) | **Moderate** | Reduced interpretability and segmentation capability | Review whether field is required; clarify collection policy |
| Clustered missingness (3+ critical fields) | 5 records (1.0%) | **High** | Records unsuitable for reliable credit evaluation | Flag for manual review; implement minimum-completeness thresholds at intake |
| Sparse spending categories | 84–99% missing per category (by design) | **Low** | Limited modelling signal but no governance breach | Treat as optional features; document sparsity for downstream users |
| Missing `ssn` and `ip_address` | 5 records each (1.0%) | **Moderate** | Reduced traceability for affected applicants | Enforce mandatory fields for identity verification |

**Overall completeness risk: Moderate** The core credit decision fields are nearly complete and conditional logic is consistent. The primary risks arise from empty strings masking true missingness, the `annual_income`/`annual_salary` schema split, and structural gaps in metadata and traceability fields.

---

## 6. Consistency Assessment

This section evaluates the internal coherence, structural integrity, and standardized representation of data across records and fields. The objective is to identify inconsistencies that may undermine data reliability, auditability, and downstream analytical integrity.

Consistency is assessed with particular focus on:

- Record-level uniqueness of application identifiers
- Standardized categorical encodings (e.g., gender representations)
- Date format and temporal consistency across timestamp fields
- Alignment between declared data types and actual stored formats
- Logical coherence between related attributes within individual records

### 6.1 Record Uniqueness

In [71]:
# Record uniqueness checks
row_duplicates = df.duplicated().sum()
id_duplicates = df["id"].duplicated().sum()

uniqueness_overview = pd.DataFrame({
    "Metric": ["Row-level duplicates", "Duplicate application IDs"],
    "Count": [row_duplicates, id_duplicates]
})

print("Record Uniqueness Overview:")
display(uniqueness_overview.style.hide(axis="index"))

Record Uniqueness Overview:


Metric,Count
Row-level duplicates,0
Duplicate application IDs,2


In [72]:
# Investigate records with duplicate application IDs to assess structural consistency
dup_ids = df[df["id"].isin(df[df["id"].duplicated()]["id"])]

print("Duplicate Application ID Records:")
display(dup_ids.sort_values(["id", "processing_timestamp"])[[
    "id", "processing_timestamp", "email", "ssn", "loan_approved", "approved_amount", "interest_rate"
]])

Duplicate Application ID Records:


,id,processing_timestamp,email,ssn,loan_approved,approved_amount,interest_rate
383,app_001,NaN,stephanie.nguyen47@mail.com,427-90-1892,False,NaN,NaN
455,app_001,NaN,stephanie.nguyen47@mail.com,NaN,False,NaN,NaN
8,app_042,NaN,joseph.lopez1@gmail.com,652-70-5530,False,NaN,NaN
354,app_042,NaN,joseph.lopez1@gmail.com,652-70-5530,False,NaN,NaN


**Findings**

No full row-level duplicates were detected (0 records). However, 2 application IDs are duplicated, affecting 4 records in total. In both cases the records are largely identical, with only minor attribute differences (e.g., one instance missing an `ssn` value). This confirms a key-level uniqueness violation rather than a full structural duplication.

From a governance perspective, this represents a high-severity issue because primary keys must be strictly unique to ensure auditability, traceability, and correct downstream processing. Even if the records are similar, duplicated identifiers can lead to double-counting, ambiguous decision histories, incorrect model training signals, and flawed reporting.

Remediation should enforce strict ID uniqueness at ingestion. This may include:
(1) implementing a primary key constraint in the data pipeline,
(2) automatically flagging duplicate IDs for manual review, and
(3) applying a deterministic deduplication rule (e.g., retain the most recent record based on `processing_timestamp`) before analytical use.

### 6.2 Categorical Encoding

In [73]:
# Analyze categorical encoding consistency for gender
gender_counts = (
    df["gender"]
    .fillna("NaN")
    .replace("", "Empty String")
    .value_counts(dropna=False)
    .reset_index()
)

gender_counts.columns = ["Gender Value", "Count"]

print("Gender Encoding Overview:")
display(gender_counts.style.hide(axis="index"))

Gender Encoding Overview:


Gender Value,Count
Male,195
Female,193
F,58
M,53
NaN,3


**Findings**

The `gender` field exhibits inconsistent categorical encoding. While the majority of records use the standardized labels Male (195) and Female (193), 58 records use F and 53 use M, representing alternative encodings of the same categories. Additionally, 2 records contain empty strings and 1 record contains a true NaN value.

This confirms a representation-level consistency issue rather than substantive data loss. From a governance perspective, this is a moderate-severity issue: inconsistent categorical encodings can distort aggregations, bias fairness analyses, and introduce unintended fragmentation in modeling pipelines. Standardization through controlled value mapping is required to ensure semantic consistency across records.

### 6.3 Data Format Consistency

In [74]:
# Analyze date format patterns in date_of_birth
import re

dob_values = df["date_of_birth"].dropna()

format_patterns = {
    "YYYY-MM-DD": r"^\d{4}-\d{2}-\d{2}$",
    "DD/MM/YYYY": r"^\d{2}/\d{2}/\d{4}$",
    "YYYY/MM/DD": r"^\d{4}/\d{2}/\d{2}$",
}

format_counts = {}
for label, pattern in format_patterns.items():
    format_counts[label] = dob_values.apply(lambda x: bool(re.match(pattern, str(x)))).sum()

unmatched = len(dob_values) - sum(format_counts.values())
format_counts["Other / Unmatched"] = unmatched

format_overview = pd.DataFrame(
    format_counts.items(),
    columns=["Date Format", "Count"]
)

format_overview["% of Non-Null"] = (format_overview["Count"] / len(dob_values) * 100).round(2)

print(f"Date format distribution for date_of_birth ({len(dob_values)} non-null values):")
display(format_overview.style.hide(axis="index").format({"% of Non-Null": "{:.2f}"}))

Date format distribution for date_of_birth (497 non-null values):


Date Format,Count,% of Non-Null
YYYY-MM-DD,340,68.41
DD/MM/YYYY,101,20.32
YYYY/MM/DD,56,11.27
Other / Unmatched,0,0.00


**Findings**

The `date_of_birth` field uses three distinct date formats across 497 non-null records: `YYYY-MM-DD` (340 records, 68.4%), `DD/MM/YYYY` (101 records, 20.3%), and `YYYY/MM/DD` (56 records, 11.3%). While all values are parseable by `pd.to_datetime`, the presence of three coexisting formats indicates a lack of standardized date handling at ingestion.

From a governance perspective, this is a moderate-severity consistency issue. Mixed date formats increase the risk of silent parsing errors — particularly for ambiguous dates where day and month values are both ≤ 12 (e.g., `03/04/1990` could be interpreted as March 4th or April 3rd). Deterministic format enforcement using ISO 8601 (`YYYY-MM-DD`) at ingestion is recommended to eliminate ambiguity and ensure reliable age calculations in downstream analyses.

### 6.4 Data Type Consistency

In [75]:
# Check for mixed Python types within each column

mixed_type_results = []

for col in df.columns:
    types_in_column = df[col].dropna().map(type).nunique()
    mixed_type_results.append({
        "Column": col,
        "Unique Python Types": types_in_column,
        "Mixed Types": types_in_column > 1
    })

mixed_type_overview = pd.DataFrame(mixed_type_results)

print("Mixed Type Consistency Overview:")
display(mixed_type_overview[mixed_type_overview["Mixed Types"] == True].style.hide(axis="index"))

Mixed Type Consistency Overview:


Column,Unique Python Types,Mixed Types
annual_income,3,True


In [76]:
# Inspect actual Python types in annual_income
type_breakdown = (
    df["annual_income"]
    .dropna()
    .map(lambda x: type(x).__name__)
    .value_counts()
    .reset_index()
)

type_breakdown.columns = ["Python Type", "Count"]

print("Type Breakdown for annual_income:")
display(type_breakdown.style.hide(axis="index"))

Type Breakdown for annual_income:


Python Type,Count
int,488
str,8
float,1


**Findings**

The type consistency check identified 1 column with mixed Python types: annual_income. The column contains 488 int values, 8 str values, and 1 float value, confirming heterogeneous storage within a single field.

All other columns are internally type-consistent.

From a governance perspective, this represents a moderate consistency issue, as mixed types within a financial column can lead to implicit casting, aggregation instability, or processing errors. Deterministic type enforcement at ingestion is recommended to ensure homogeneous storage going forward.

### 6.5 Consistency Risk Assessment

| Finding | Evidence | Severity | Governance Impact | Recommended Control |
|---|---|---|---|---|
| Duplicate application IDs | 2 duplicated IDs affecting 4 records | **High** | Violates primary key uniqueness; risk of double-counting, audit ambiguity, and unstable joins | Enforce primary key constraint; flag duplicates; apply deterministic deduplication rule |
| Inconsistent gender encoding (Male/M, Female/F) | 111 records using non-standard abbreviations (58 F, 53 M) | **Moderate** | Fragmented aggregation; distorted fairness analysis; inconsistent categorical reporting | Standardize categorical values via controlled mapping at ingestion |
| Mixed Python types in `annual_income` | 488 int, 8 str, 1 float | **Moderate** | Implicit casting risk; aggregation instability; inconsistent numeric processing | Enforce deterministic numeric casting prior to storage; reject non-numeric inputs |
| Minor attribute-level inconsistencies within duplicate IDs | Missing `ssn` in 1 duplicated record instance | **Low–Moderate** | Potential synchronization issue across systems | Implement record reconciliation logic during ingestion |
| Inconsistent date formats in `date_of_birth` | 3 coexisting formats: YYYY-MM-DD (340), DD/MM/YYYY (101), YYYY/MM/DD (56) | **Moderate** | Risk of silent parsing errors for ambiguous dates; inconsistent temporal processing | Enforce ISO 8601 (YYYY-MM-DD) at ingestion |

**Overall consistency risk: Moderate** 

While most columns are structurally stable, primary key duplication represents a high-severity governance breach. Representation-level inconsistencies (categorical encoding, mixed types within annual_income or inconsistent date formats) introduce moderate analytical risk. Strengthening key constraints and enforcing deterministic ingestion controls is recommended to ensure long-term structural integrity.

---

## 7. Validity Assessment

This section evaluates whether recorded values comply with defined business rules, domain constraints, and syntactic standards. While completeness and consistency assess structural integrity, validity focuses on whether the data itself is logically plausible and conforms to expected formats and numerical boundaries.

Validity is assessed with particular focus on:

- Numerical range validation of financial variables (e.g., non-negative income, bounded debt-to-income ratios, non-negative interest rates)
- Plausibility checks on aggregated spending behavior relative to reported income
- Format validation of identity and contact fields (e.g., SSN structure, ZIP code format, email syntax)
- Alignment of stored data types with domain expectations for financial, temporal, and decision-related attributes

The objective is to identify rule violations that could distort credit risk assessment, bias model training, or compromise regulatory compliance.

### 7.1 Numerical Range Validation

In [77]:
# Validate numerical business rule constraints for key financial variables
income = pd.to_numeric(df["annual_income"], errors="coerce")

neg_income = (income < 0).sum()
bad_dti = ((df["debt_to_income"] < 0) | (df["debt_to_income"] > 1)).sum()
neg_ir = (df["interest_rate"] < 0).sum()

range_validation_overview = pd.DataFrame({
    "Metric": [
        "Negative annual_income",
        "DTI outside [0,1]",
        "Negative interest_rate"
    ],
    "Count": [
        neg_income,
        bad_dti,
        neg_ir
    ]
})

print("Numerical Range Validation Overview:")
display(range_validation_overview.style.hide(axis="index"))

Numerical Range Validation Overview:


Metric,Count
Negative annual_income,0
"DTI outside [0,1]",1
Negative interest_rate,0


In [78]:
# Identify records with invalid DTI and display only relevant fields
invalid_dti_records = df[
    (df["debt_to_income"] < 0) | (df["debt_to_income"] > 1)
][[
    "id",
    "annual_income",
    "debt_to_income",
    "approved_amount",
    "interest_rate",
    "loan_approved"
]]

print("Records with invalid DTI (relevant fields only):")
display(invalid_dti_records)

Records with invalid DTI (relevant fields only):


,id,annual_income,debt_to_income,approved_amount,interest_rate,loan_approved
316,app_402,88000,1.85,17000.0,3.2,True


In [79]:
# Validate numeric parseability of annual_income values
invalid_income_format = df["annual_income"].notna().sum() - income.notna().sum()

format_validation_overview = pd.DataFrame({
    "Metric": ["Non-numeric annual_income values"],
    "Count": [invalid_income_format]
})

print("Income Format Validation Overview:")
display(format_validation_overview.style.hide(axis="index"))

Income Format Validation Overview:


Metric,Count
Non-numeric annual_income values,0


In [80]:
# Validate plausibility of spending values across all spending categories
spend_cols = [c for c in df.columns if c.startswith("spending_")]

neg_spending = (df[spend_cols] < 0).sum().sum()
extreme_spending = (df[spend_cols] > 1_000_000).sum().sum()

spending_validation_overview = pd.DataFrame({
    "Metric": [
        "Negative spending values",
        "Spending values > 1,000,000"
    ],
    "Count": [
        neg_spending,
        extreme_spending
    ]
})

print("Spending Plausibility Validation Overview:")
display(spending_validation_overview.style.hide(axis="index"))

Spending Plausibility Validation Overview:


Metric,Count
Negative spending values,0
"Spending values > 1,000,000",0


In [81]:
# validate range constraints for credit_history_months and savings_balance
invalid_credit_history = (df["credit_history_months"] < 0).sum()
invalid_savings = (df["savings_balance"] < 0).sum()

range_validation_additional = pd.DataFrame({
    "Field": ["credit_history_months", "savings_balance"],
    "Constraint": ["≥ 0", "≥ 0"],
    "Violations": [invalid_credit_history, invalid_savings],
    "% of Total": [
        f"{invalid_credit_history/len(df)*100:.1f}%",
        f"{invalid_savings/len(df)*100:.1f}%"
    ]
})

print("Additional Range Validation:")
display(range_validation_additional.style.hide(axis="index"))

Additional Range Validation:


Field,Constraint,Violations,% of Total
credit_history_months,≥ 0,2,0.4%
savings_balance,≥ 0,1,0.2%


In [82]:
# inspect records with negative credit_history_months or savings_balance
neg_credit = df[df["credit_history_months"] < 0][["id", "credit_history_months", "annual_income", "loan_approved"]]
neg_savings = df[df["savings_balance"] < 0][["id", "savings_balance", "annual_income", "loan_approved"]]

print(f"Records with negative credit_history_months:")
display(neg_credit)

print(f"\nRecords with negative savings_balance:")
display(neg_savings)

Records with negative credit_history_months:


,id,credit_history_months,annual_income,loan_approved
137,app_043,-10,131000,True
162,app_156,-3,25000,False



Records with negative savings_balance:


,id,savings_balance,annual_income,loan_approved
159,app_290,-5000,104000,True


**Findings**

The numerical range validation identified 1 rule violation (0.2%) in `debt_to_income`. The affected record (id = app_402) reports an `annual_income` of 88,000 and a `debt_to_income` value of 1.85, while the loan was approved with an `approved_amount` of 17,000 and an `interest_rate` of 3.2. A DTI of 1.85 exceeds the defined domain constraint (0 ≤ DTI ≤ 1) and indicates either a calculation error or a missing validation rule at intake.

Additionally, `credit_history_months` contains 2 negative values (0.4%), with a minimum of −10. As credit history duration cannot be negative, these values are logically impossible and likely indicate data entry errors. `savings_balance` contains 1 negative value (0.2%), with a minimum of −5,000, which may indicate a data entry error or an undocumented overdraft representation.

No negative values were detected for `annual_income` (0 records) or `interest_rate` (0 records), and no implausible spending values were identified (0 negative, 0 extreme), suggesting that numerical controls are generally effective.

From a governance perspective, the DTI violation is the most critical finding, as the affected field is decision-critical and the application was approved despite an out-of-range value, indicating a control gap in business rule enforcement. The negative values in `credit_history_months` and `savings_balance` are lower in frequency but equally indicative of missing ingestion-level constraints. Strict non-negativity and domain validation rules should be implemented at ingestion for all three fields.

### 7.2 Format Validation

In [83]:
# Validate that SSN values follow the required XXX-XX-XXXX format
import re

invalid_ssn = df["ssn"].dropna().apply(
    lambda x: not bool(re.fullmatch(r"\d{3}-\d{2}-\d{4}", str(x)))
).sum()

ssn_format_validation_overview = pd.DataFrame({
    "Metric": ["Invalid SSN format (not XXX-XX-XXXX)"],
    "Count": [invalid_ssn]
})

print("SSN Format Validation Overview:")
display(ssn_format_validation_overview.style.hide(axis="index"))

SSN Format Validation Overview:


Metric,Count
Invalid SSN format (not XXX-XX-XXXX),0


In [84]:
# Validate that ZIP codes contain only numeric characters
invalid_zip = df["zip_code"].dropna().apply(
    lambda x: not str(x).isdigit()
).sum()

zip_format_validation_overview = pd.DataFrame({
    "Metric": ["Invalid ZIP code format (non-numeric characters)"],
    "Count": [invalid_zip]
})

print("ZIP Code Format Validation Overview:")
display(zip_format_validation_overview.style.hide(axis="index"))

ZIP Code Format Validation Overview:


Metric,Count
Invalid ZIP code format (non-numeric characters),0


In [85]:
# Validate that email values contain the required "@" symbol
invalid_email = df["email"].dropna().apply(
    lambda x: "@" not in str(x)
).sum()

email_format_validation_overview = pd.DataFrame({
    "Metric": ["Invalid email format (missing '@')"],
    "Count": [invalid_email]
})

print("Email Format Validation Overview:")
display(email_format_validation_overview.style.hide(axis="index"))

Email Format Validation Overview:


Metric,Count
Invalid email format (missing '@'),1


In [86]:
# Identify records with invalid email format (missing "@")
invalid_email_records = df[
    df["email"].notna() & 
    (~df["email"].astype(str).str.contains("@"))
][[
    "id",
    "full_name",
    "email",
    "zip_code",
    "loan_approved"
]]

print("Records with invalid email format:")
display(invalid_email_records)

Records with invalid email format:


,id,full_name,email,zip_code,loan_approved
181,app_299,Samuel Gonzalez,test.user.outlook.com,10054,False


**Findings**

The format validation identified 1 format-related issue across identity and contact attributes. No invalid `ssn` values were detected, confirming consistent adherence to the required XXX-XX-XXXX structure.

The single violation stems from the `email` field: 1 record contains a syntactically invalid string without the required "@" symbol. The remaining records with missing email values are already accounted for in the completeness assessment. Notably, the affected application was approved, indicating that contact field validation is not enforced prior to decision processing.

From a governance perspective, this represents a low-to-moderate severity issue. While the frequency is minimal, the presence of a malformed email in an approved application indicates a gap in intake validation controls, weakening traceability and communication reliability.

Recommended remediation includes enforcing regex-based email format validation at ingestion and implementing deterministic validation rules prior to approval workflows.

### 7.3 Data Type Validation

In [87]:
# Validate that critical fields match expected data types
expected_types = {
    "id": "object",
    "processing_timestamp": "datetime64[ns]",
    "date_of_birth": "datetime64[ns]",
    "annual_income": "float64",
    "debt_to_income": "float64",
    "interest_rate": "float64",
    "approved_amount": "float64",
    "loan_approved": "bool"
}

dtype_validation_results = []

for col, expected in expected_types.items():
    if col in df.columns:
        actual = str(df[col].dtype)
        dtype_validation_results.append({
            "Field": col,
            "Expected Type": expected,
            "Actual Type": actual,
            "Matches Expected": actual == expected
        })

dtype_validation_overview = pd.DataFrame(dtype_validation_results)

print("Data Type Validation Overview:")
display(dtype_validation_overview)

Data Type Validation Overview:


,Field,Expected Type,Actual Type,Matches Expected
0,id,object,str,False
1,processing_timestamp,datetime64[ns],str,False
2,date_of_birth,datetime64[ns],str,False
3,annual_income,float64,object,False
4,debt_to_income,float64,float64,True
5,interest_rate,float64,float64,True
6,approved_amount,float64,float64,True
7,loan_approved,bool,bool,True


**Findings**

The data type validation identified 2 out of 8 assessed fields with mismatched types. `processing_timestamp` is stored as object instead of datetime64[ns], and `annual_income` is stored as object instead of float64.

The remaining 6 fields (75%) match their expected domain types, including all core financial and decision variables.

From a governance perspective, this represents a moderate-severity issue, as misaligned types in temporal and financial attributes may lead to unstable processing, aggregation inconsistencies, or weakened audit controls. Deterministic type enforcement at ingestion is recommended.

### 7.4 Validity Risk Assessment

| Finding | Evidence | Severity | Governance Impact | Recommended Control |
|---|---|---|---|---|
| DTI outside valid domain [0,1] | 1 record (0.2%) – `debt_to_income` = 1.85 in approved application | **Moderate** | Decision-critical metric violates domain constraint; indicates control gap in approval logic | Enforce strict range validation (0 ≤ DTI ≤ 1) at ingestion or calculate server-side |
| Negative `credit_history_months` | 2 records (0.4%) – minimum value of −10 | **Moderate** | Logically impossible values in decision-critical field; distorts creditworthiness assessment | Enforce non-negativity constraint (≥ 0) at ingestion |
| Negative `savings_balance` | 1 record (0.2%) – minimum value of −5,000 | **Moderate** | Implausible balance value; may distort affordability assessment | Enforce non-negativity constraint (≥ 0) at ingestion; clarify whether negative values represent overdrafts |
| Invalid email format | 1 record (0.2%) – syntactically invalid string missing `@`; approved application affected | **Low–Moderate** | Weak traceability and communication reliability; intake validation not enforced | Implement mandatory email validation (regex-based) at ingestion |
| Data type mismatches (`processing_timestamp`, `annual_income`) | 2 of 8 assessed fields (25%) misaligned | **Moderate** | Risk of unstable processing, aggregation errors, weakened audit controls | Enforce deterministic casting and schema validation rules at ingestion |

**Overall validity risk: Moderate** The dataset shows generally strong adherence to domain and format rules, with violations limited in frequency. However, the DTI domain breach in an approved application, negative values in two decision-critical fields, and persistent type mismatches indicate systematic gaps in ingestion-level validation controls. Strengthening deterministic rule enforcement and domain constraints at ingestion is recommended to ensure regulatory robustness and analytical stability.

---

## 8. Accuracy Assessment
This section evaluates whether recorded values are substantively plausible and logically coherent in the context of credit decision-making. While prior sections assessed structural integrity and rule compliance, this assessment focuses on whether the data reflects realistic real-world conditions.

Accuracy and plausibility are examined across four dimensions:
- Realistic demographic characteristics
- Financial behavior in relation to economic capacity
- Logical alignment between risk indicators and decision outcomes
- Reasonableness of pricing and contractual terms

The objective is to identify records that are technically valid but substantively implausible, as such inconsistencies may distort risk assessment, bias analytical models, or weaken the credibility of automated decision processes.

### 8.1 Age Plausibility

In [88]:
# Validate plausibility of applicant age based on date of birth

# Using a fixed reference date for reproducibility (date of analysis creation)
reference_date = pd.Timestamp("2025-02-26")

# format="mixed" is required — the dataset contains three coexisting date formats:
# YYYY-MM-DD (68.4%), DD/MM/YYYY (20.3%), YYYY/MM/DD (11.3%).
# Without it, pandas infers a single format and silently converts the rest to NaT.
df["date_of_birth_parsed"] = pd.to_datetime(
    df["date_of_birth"], format="mixed", dayfirst=False, errors="coerce"
)
df["age"] = (reference_date - df["date_of_birth_parsed"]).dt.days // 365

unrealistic_age = ((df["age"] < 18) | (df["age"] > 100)).sum()

age_plausibility_overview = pd.DataFrame({
    "Metric": ["Age outside plausible range (18–100)"],
    "Count": [unrealistic_age]
})

print(f"Age Plausibility Validation Overview (reference date: {reference_date.date()}):")
display(age_plausibility_overview.style.hide(axis="index"))

Age Plausibility Validation Overview (reference date: 2025-02-26):


Metric,Count
Age outside plausible range (18–100),0


**Findings**

The age plausibility check identified 0 records (0%) outside the defined range of 18–100 years. All applicants fall within a realistic demographic spectrum, indicating no age-related accuracy concerns.

### 8.2 Spending vs. Income Plausibility

In [89]:
# Validate plausibility of total spending relative to reported income
# Note: spending categories represent monthly amounts; income is annual, so we annualize spending for comparison

spend_cols = [c for c in df.columns if c.startswith("spending_")]
df["total_monthly_spending"] = df[spend_cols].sum(axis=1)
df["total_annual_spending"] = df["total_monthly_spending"] * 12

income = pd.to_numeric(df["annual_income"], errors="coerce")

implausible_spending = (df["total_annual_spending"] > income).sum()

spending_income_plausibility_overview = pd.DataFrame({
    "Metric": ["Annualized spending exceeds reported income"],
    "Count": [implausible_spending]
})

print("Spending vs Income Plausibility Overview:")
display(spending_income_plausibility_overview.style.hide(axis="index"))

Spending vs Income Plausibility Overview:


Metric,Count
Annualized spending exceeds reported income,1


In [90]:
# Identify records where annualized spending exceeds reported income
implausible_spending_records = df[
    df["total_annual_spending"] > income
][[
    "id",
    "annual_income",
    "total_monthly_spending",
    "total_annual_spending",
    "loan_approved",
    "approved_amount",
    "interest_rate"
]]

print("Records with annualized spending exceeding reported income:")
display(implausible_spending_records)

Records with annualized spending exceeding reported income:


,id,annual_income,total_monthly_spending,total_annual_spending,loan_approved,approved_amount,interest_rate
426,app_190,0,1389.0,16668.0,False,NaN,NaN


**Findings**

When annualizing monthly spending (×12) for a like-for-like comparison with annual income, 1 record shows total annualized spending exceeding reported income. The affected applicant (app_190) reports an `annual_income` of 0, while total annualized spending amounts to approximately 16,668. The loan was not approved.

This indicates a high-severity plausibility issue. Either income was incorrectly recorded as zero, income is missing but represented as 0 instead of null, or spending data is misclassified. From a governance perspective, zero-income values combined with positive spending distort affordability assessments and risk modelling.

Remediation should include validation rules preventing zero-income records with positive spending, and a distinction between true zero values and missing financial information at ingestion.

### 8.3 Decision Logic Plausibility

In [91]:
# Validate decision plausibility: approved loans with zero or missing income
approved_zero_income = (
    (df["loan_approved"] == True) &
    ((income <= 0) | income.isna())
).sum()

decision_plausibility_overview = pd.DataFrame({
    "Metric": ["Approved loans with zero or missing income"],
    "Count": [approved_zero_income]
})

print("Decision Logic Plausibility Overview:")
display(decision_plausibility_overview.style.hide(axis="index"))

Decision Logic Plausibility Overview:


Metric,Count
Approved loans with zero or missing income,3


In [92]:
# Identify approved loans with zero or missing income
approved_zero_income_records = df[
    (df["loan_approved"] == True) &
    ((income <= 0) | income.isna())
][[
    "id",
    "annual_income",
    "total_monthly_spending",
    "approved_amount",
    "interest_rate",
    "debt_to_income"
]]

print("Approved loans with zero or missing income (relevant fields only):")
display(approved_zero_income_records)

Approved loans with zero or missing income (relevant fields only):


,id,annual_income,total_monthly_spending,approved_amount,interest_rate,debt_to_income
76,app_436,NaN,339.0,63000.0,3.9,0.13
94,app_421,NaN,1424.0,45000.0,6.0,0.06
149,app_449,NaN,498.0,54000.0,5.7,0.07


**Findings**

There are 3 approved loans where `annual_income` is NaN at the point of decision. Despite the absence of income in the documented field, these applicants received approved amounts between 45,000 and 63,000, with interest rates ranging from 3.9% to 6.0% and low debt-to-income ratios (0.06–0.13).

Notably, all 3 records correspond to applicants whose income was recorded under the undocumented `annual_salary` field rather than `annual_income` (as identified in Section 5.3). After reconciliation into `annual_income_clean`, these records do contain valid income values (45,000–75,000). The underlying data is therefore not missing but the approval pipeline appears to have processed these applications without referencing the correct income field, or without validating that income was present in the expected location.

From a governance perspective, this constitutes a high-severity decision logic issue. Regardless of whether income existed elsewhere in the record, the approval was granted without the standard income field being populated, which undermines affordability assessment, auditability, and regulatory defensibility. Remediation should include a mandatory validation rule that blocks approval when the canonical income field is missing or non-positive, combined with schema reconciliation to ensure all income data flows into a single standardized field before decision processing.

### 8.4 Interest Rate Plausibility

In [93]:
# Validate plausibility of interest rate values
# Define plausible interest rate range (e.g., 0%–25%)
invalid_interest_rate = (
    (df["interest_rate"] < 0) |
    (df["interest_rate"] > 25)
).sum()

interest_rate_plausibility_overview = pd.DataFrame({
    "Metric": ["Interest rate outside plausible range (0–25%)"],
    "Count": [invalid_interest_rate]
})

print("Interest Rate Plausibility Overview:")
display(interest_rate_plausibility_overview.style.hide(axis="index"))

Interest Rate Plausibility Overview:


Metric,Count
Interest rate outside plausible range (0–25%),0


**Findings**

No records (0 cases) show interest rates outside the defined plausible range of 0–25%. All observed rates fall within commercially realistic lending boundaries. No pricing anomalies or extreme values were detected, and the variable appears structurally stable. Ongoing range validation at ingestion is sufficient to maintain governance control in this area.

### 8.5 Accuracy Risk Assessment

| Finding | Evidence | Severity | Governance Impact | Recommended Control |
|---|---|---|---|---|
| Age outside plausible range | 0 records | **Low** | No demographic plausibility risk detected | Maintain range validation (18–100) at ingestion |
| Total spending exceeds reported income | 1 record (income = 0, spending = 1,389) | **High** | Financial inconsistency; distorted affordability assessment | Enforce rule preventing positive spending with zero income; distinguish zero vs missing income |
| Approved loans with zero or missing income | 3 approved loans | **High** | Critical underwriting breach; reduced audit defensibility | Block approval when income is missing/non-positive; implement mandatory completeness checks before decision |
| Interest rate outside plausible range (0–25%) | 0 records | **Low** | No pricing anomalies detected | Maintain deterministic range validation |

**Overall accuracy risk: Moderate–High** While demographic and pricing variables are stable, the presence of decision approvals without documented income and one spending-income contradiction represent material plausibility breaches. Strengthening hard validation rules in underwriting logic is required to ensure decision integrity and regulatory defensibility.

---

## 9. Consolidated Risk Assessment

This section synthesises findings from all four quality dimensions into a unified risk profile for the NovaCred dataset, enabling prioritised governance action.

| Dimension | Finding | Affected Records | Severity |
|---|---|---|---|
| **Completeness** | Empty strings masking missingness (`email`, `gender`, `date_of_birth`, `zip_code`) | 14 values | **High** |
| **Completeness** | Missing `processing_timestamp` | 440/502 (87.6%) | **High** |
| **Completeness** | Clustered missingness (3+ critical fields missing per record) | 5 (1.0%) | **High** |
| **Completeness** | `annual_income` / `annual_salary` schema split | 5 (1.0%) | **Moderate** |
| **Completeness** | Missing `loan_purpose` | 452/502 (90.0%) | **Moderate** |
| **Completeness** | Missing `ssn` and `ip_address` | 5 each (1.0%) | **Moderate** |
| **Consistency** | Duplicate application IDs | 4 (0.8%) | **High** |
| **Consistency** | Inconsistent gender encoding (`M`/`Male`, `F`/`Female`) | 111 (22.1%) | **Moderate** |
| **Consistency** | Mixed Python types in `annual_income` | 9 (1.8%) | **Moderate** |
| **Consistency** | Inconsistent date formats in `date_of_birth` | 3 formats across 497 records | **Moderate** |
| **Validity** | DTI outside valid domain [0, 1] | 1 (0.2%) | **Moderate** |
| **Validity** | Negative `credit_history_months` | 2 (0.4%) | **Moderate** |
| **Validity** | Negative `savings_balance` | 1 (0.2%) | **Moderate** |
| **Validity** | Invalid email format (missing `@`) | 1 (0.2%) | **Low–Moderate** |
| **Validity** | Data type mismatches (`processing_timestamp`, `annual_income`) | 2 fields | **Moderate** |
| **Accuracy** | Approved loans with zero or missing income | 3 (1.0%) | **High** |
| **Accuracy** | Total spending exceeds reported income (income = 0) | 1 (0.2%) | **High** |

**Risk summary by dimension:**

| Dimension | Overall Risk | Key Driver |
|---|---|---|
| **Completeness** | **Moderate** | Empty string masking, audit trail gaps, schema inconsistency |
| **Consistency** | **Moderate** | Primary key duplication, categorical encoding fragmentation |
| **Validity** | **Moderate** | DTI domain violation, negative values in decision-critical fields, type mismatches |
| **Accuracy** | **Moderate–High** | Loan approvals without income; spending–income contradiction |

**Overall dataset risk: Moderate–High**

The dataset is structurally sound in its core decision fields — financial variables are nearly complete, conditional decision logic is consistent, and most format validations pass. However, multiple high-severity issues exist in areas that directly affect auditability, underwriting defensibility, and governance compliance:

1. **Three loans were approved without documented income** — a critical underwriting control failure with direct regulatory exposure.
2. **Duplicate application IDs** violate primary key integrity and compromise traceability and audit reliability.
3. **Empty strings silently masking missing values** cause true missingness to be systematically undercounted in any standard analysis.
4. **87.6% of records lack a processing timestamp**, severely weakening the audit trail for the credit decision lifecycle.

These findings are sufficient to conclude that the dataset, in its current state, does not meet the minimum quality bar required for reliable automated credit decision-making without remediation.

---

## 10. Recommended Controls and Remediation Plan

This section translates the findings from the consolidated risk assessment into actionable remediation steps. It is structured in two parts: a set of code demonstrations showing how key issues can be addressed programmatically, followed by a prioritised action plan for governance and engineering teams.

### 10.1 Remediation Demonstrations

In [94]:
# reconcile annual_income / annual_salary schema split into a single canonical field
df["annual_income_clean"] = pd.to_numeric(
    df["annual_income"].combine_first(df["annual_salary"]),
    errors="coerce"
)

reconciled = df["annual_income_clean"].notna().sum()
missing = df["annual_income_clean"].isna().sum()
print(f"annual_income_clean — populated: {reconciled}/502 ({reconciled/502*100:.1f}%), missing: {missing}/502 ({missing/502*100:.1f}%)")

annual_income_clean — populated: 502/502 (100.0%), missing: 0/502 (0.0%)


In [95]:
# standardise gender encoding via controlled value mapping
gender_map = {"M": "Male", "F": "Female"}
df["gender_clean"] = df["gender"].replace(gender_map)

print("Gender value counts after standardisation:")
print(df["gender_clean"].value_counts(dropna=False).to_string())

Gender value counts after standardisation:
gender_clean
Female    251
Male      248
NaN         3


In [96]:
# deduplicate on application ID, retaining the most complete record per duplicate
df["_completeness_score"] = df[["ssn", "email", "ip_address", "gender", "date_of_birth", "zip_code"]].notna().sum(axis=1)
df_deduped = (
    df.sort_values("_completeness_score", ascending=False)
      .drop_duplicates(subset="id", keep="first")
      .drop(columns="_completeness_score")
      .reset_index(drop=True)
)
print(f"Records before deduplication: {len(df)}")
print(f"Records after deduplication:  {len(df_deduped)}")
print(f"Duplicates removed:           {len(df) - len(df_deduped)}")

Records before deduplication: 502
Records after deduplication:  500
Duplicates removed:           2


In [97]:
# flag records failing minimum completeness threshold (3+ critical fields missing)
critical_fields = ["annual_income", "ssn", "ip_address", "email", "gender", "date_of_birth", "zip_code"]
df_deduped["_missing_critical"] = df_deduped[critical_fields].isna().sum(axis=1)
df_deduped["_flag_incomplete"] = df_deduped["_missing_critical"] >= 3

flagged = df_deduped["_flag_incomplete"].sum()
print(f"Records flagged for clustered missingness (3+ critical fields): {flagged} ({flagged/len(df_deduped)*100:.1f}%)")
display(df_deduped[df_deduped["_flag_incomplete"]][["id"] + critical_fields])

Records flagged for clustered missingness (3+ critical fields): 4 (0.8%)


,id,annual_income,ssn,ip_address,email,gender,date_of_birth,zip_code
496,app_165,66000,NaN,NaN,NaN,Male,NaN,10077
497,app_120,103000,NaN,NaN,NaN,Female,NaN,90240
498,app_268,112000,NaN,NaN,NaN,NaN,1971-05-14,10005
499,app_075,61000,NaN,NaN,NaN,NaN,NaN,NaN


In [98]:
# flag approved loans with zero or missing income (using original field to demonstrate pre-remediation issue)
income_original = pd.to_numeric(df_deduped["annual_income"], errors="coerce")
df_deduped["_flag_approved_no_income"] = (
    (df_deduped["loan_approved"] == True) &
    ((income_original <= 0) | income_original.isna())
)

flagged = df_deduped["_flag_approved_no_income"].sum()
print(f"Approved loans with missing/zero income (pre-remediation): {flagged} ({flagged/df_deduped['loan_approved'].sum()*100:.1f}% of approved)")
display(df_deduped[df_deduped["_flag_approved_no_income"]][["id", "annual_income", "annual_income_clean", "approved_amount", "interest_rate", "debt_to_income"]])

Approved loans with missing/zero income (pre-remediation): 3 (1.0% of approved)


,id,annual_income,annual_income_clean,approved_amount,interest_rate,debt_to_income
78,app_436,NaN,45000.0,63000.0,3.9,0.13
92,app_421,NaN,46000.0,45000.0,6.0,0.06
133,app_449,NaN,75000.0,54000.0,5.7,0.07


In [99]:
# flag records with DTI outside valid domain [0, 1]
df_deduped["_flag_invalid_dti"] = (
    (df_deduped["debt_to_income"] < 0) | (df_deduped["debt_to_income"] > 1)
)

flagged = df_deduped["_flag_invalid_dti"].sum()
print(f"Records with DTI outside valid domain [0, 1]: {flagged} ({flagged/len(df_deduped)*100:.1f}%)")
display(df_deduped[df_deduped["_flag_invalid_dti"]][["id", "annual_income_clean", "debt_to_income", "approved_amount", "interest_rate", "loan_approved"]])

Records with DTI outside valid domain [0, 1]: 1 (0.2%)


,id,annual_income_clean,debt_to_income,approved_amount,interest_rate,loan_approved
314,app_402,88000.0,1.85,17000.0,3.2,True


In [100]:
# enforce correct data types for processing_timestamp and annual_income_clean
df_deduped["processing_timestamp"] = pd.to_datetime(df_deduped["processing_timestamp"], errors="coerce")
df_deduped["annual_income_clean"] = pd.to_numeric(df_deduped["annual_income_clean"], errors="coerce")

print("Data types after casting:")
print(df_deduped[["processing_timestamp", "annual_income_clean"]].dtypes)

Data types after casting:
processing_timestamp    datetime64[us, UTC]
annual_income_clean                 float64
dtype: object


In [101]:
# set negative credit_history_months to NaN (logically impossible values)
neg_credit_mask = df_deduped["credit_history_months"] < 0
df_deduped.loc[neg_credit_mask, "credit_history_months"] = np.nan

print(f"credit_history_months — negative values set to NaN: {neg_credit_mask.sum()} records affected")
print(f"Remaining nulls: {df_deduped['credit_history_months'].isna().sum()}")

credit_history_months — negative values set to NaN: 2 records affected
Remaining nulls: 2


In [102]:
# set negative savings_balance to NaN (no overdraft representation documented)
neg_savings_mask = df_deduped["savings_balance"] < 0
df_deduped.loc[neg_savings_mask, "savings_balance"] = np.nan

print(f"savings_balance — negative values set to NaN: {neg_savings_mask.sum()} records affected")
print(f"Remaining nulls: {df_deduped['savings_balance'].isna().sum()}")

savings_balance — negative values set to NaN: 1 records affected
Remaining nulls: 1


In [103]:
# set syntactically invalid email values (missing "@") to NaN
invalid_email_mask = df_deduped["email"].notna() & ~df_deduped["email"].astype(str).str.contains("@")
df_deduped.loc[invalid_email_mask, "email"] = np.nan

print(f"email — invalid format set to NaN: {invalid_email_mask.sum()} records affected")

email — invalid format set to NaN: 1 records affected


In [104]:
# Standardize date_of_birth to datetime.
# format="mixed" is required — three coexisting formats in the dataset
# (YYYY-MM-DD, DD/MM/YYYY, YYYY/MM/DD). Without it, pandas infers one format
# and silently converts the other two to NaT, corrupting downstream age analysis.
df_deduped["date_of_birth"] = pd.to_datetime(
    df_deduped["date_of_birth"], format="mixed", dayfirst=False, errors="coerce"
)

parsed = df_deduped["date_of_birth"].notna().sum()
print(f"date_of_birth — parsed to datetime: {parsed}/{len(df_deduped)} records")
print(f"NaT (unparseable): {df_deduped['date_of_birth'].isna().sum()}")
print(f"dtype: {df_deduped['date_of_birth'].dtype}")

date_of_birth — parsed to datetime: 496/500 records
NaT (unparseable): 4
dtype: datetime64[us]


In [105]:
# replace zero annual_income_clean with NaN to distinguish true zero from missing
zero_income_mask = df_deduped["annual_income_clean"] == 0
df_deduped.loc[zero_income_mask, "annual_income_clean"] = np.nan

print(f"annual_income_clean — zero values replaced with NaN: {zero_income_mask.sum()} records affected")

annual_income_clean — zero values replaced with NaN: 1 records affected


### 10.2 Priority Actions

The following table consolidates all recommended controls into a prioritised remediation plan. Actions are grouped into four priority tiers: P1 items address critical control failures requiring immediate intervention, P2 items cover short-term structural improvements to ingestion and validation pipelines, P3 items establish medium-term format and type enforcement, and P4 items define ongoing governance and monitoring practices.

| Priority | Action | Target Field(s) | Severity Addressed |
|---|---|---|---|
| **P1 — Immediate** | Block loan approval when `annual_income` is missing or ≤ 0 | `annual_income`, `loan_approved` | High (accuracy) |
| **P1 — Immediate** | Enforce primary key uniqueness constraint at ingestion; flag and quarantine duplicate IDs | `id` | High (consistency) |
| **P1 — Immediate** | Normalise empty strings to `NaN` at ingestion for all string fields | All string columns | High (completeness) |
| **P2 — Short-term** | Reconcile `annual_income` and `annual_salary` into a single canonical field | `annual_income`, `annual_salary` | Moderate (completeness) |
| **P2 — Short-term** | Enforce mandatory `processing_timestamp` logging for all records | `processing_timestamp` | High (completeness) |
| **P2 — Short-term** | Standardise gender encoding via controlled vocabulary at intake | `gender` | Moderate (consistency) |
| **P2 — Short-term** | Enforce strict domain constraint (0 ≤ DTI ≤ 1) at ingestion or calculate server-side | `debt_to_income` | Moderate (validity) |
| **P2 — Short-term** | Enforce non-negativity constraint (≥ 0) for `credit_history_months` at ingestion | `credit_history_months` | Moderate (validity) |
| **P2 — Short-term** | Enforce non-negativity constraint (≥ 0) for `savings_balance` at ingestion; clarify whether negative values represent overdrafts | `savings_balance` | Moderate (validity) |
| **P2 — Short-term** | Standardise `date_of_birth` to ISO 8601 format (`YYYY-MM-DD`) at ingestion | `date_of_birth` | Moderate (consistency) |
| **P3 — Medium-term** | Implement mandatory email format validation (regex-based) at intake | `email` | Low–Moderate (validity) |
| **P3 — Medium-term** | Enforce deterministic numeric casting for `annual_income` | `annual_income` | Moderate (consistency, validity) |
| **P3 — Medium-term** | Apply minimum completeness threshold (< 3 missing critical fields) before decision processing | All critical fields | High (completeness) |
| **P4 — Ongoing** | Implement schema versioning and field-level documentation for all undocumented fields | `loan_purpose`, `notes`, `annual_salary` | Moderate (governance) |
| **P4 — Ongoing** | Establish automated data quality monitoring with alerts on rule violations | All fields | Cross-cutting |

## 11. Export Cleaned Dataset

In [106]:
# Drop internal/helper columns 
cols_to_drop = [c for c in df_deduped.columns if c.startswith("_")]
df_clean = df_deduped.drop(columns=cols_to_drop).reset_index(drop=True)

print(f"Final cleaned dataset: {df_clean.shape[0]} records x {df_clean.shape[1]} columns")
print(f"Removed internal columns: {cols_to_drop}")

# Standardize dtypes for safe Parquet export

# Convert to pandas nullable dtypes
df_clean = df_clean.convert_dtypes()

# Ensure datetime columns are real datetime (not object)
for col in df_clean.columns:
    if "date" in col or "timestamp" in col:
        df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

# Convert categorical columns to string
for col in df_clean.select_dtypes(include="category"):
    df_clean[col] = df_clean[col].astype("string")

# Fix object columns with mixed types
for col in df_clean.select_dtypes(include="object"):
    df_clean[col] = df_clean[col].astype("string")

# Export
output_path = "../data/processed/cleaned_credit_applications.parquet"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df_clean.to_parquet(output_path, index=False)

print(f"\nExported to: {output_path}")

Final cleaned dataset: 500 records x 41 columns
Removed internal columns: ['_missing_critical', '_flag_incomplete', '_flag_approved_no_income', '_flag_invalid_dti']

Exported to: ../data/processed/cleaned_credit_applications.parquet
